In [7]:
from pathlib import Path
import pandas as pd

import pickle
DATA_DIR = Path("/root/capsule/data")
SCRATCH_DIR = Path("/root/capsule/scratch")
MATCH_DIR = SCRATCH_DIR / "ophys_transcriptomics_match_tables"
MATCH_DIR.mkdir(exist_ok=True)

load data information

In [ ]:
data_info = pickle.load(open('data_info_hcr.pkl', 'rb'))
mouse_ids = list(data_info.keys())
print('mouse_ids:', mouse_ids)

mouse_ids: [755252, 767018, 767022, 782149, 788406, 790322]


# unifed table connecting ophys sessions and xenium

In [8]:

for mouse_id in mouse_ids:
    ophys_session_names = {asset_name: data_info[mouse_id]['ophys'][asset_name]['raw'] for asset_name in data_info[mouse_id]['ophys'] if data_info[mouse_id]['ophys'][asset_name]['session_type'] == 'STAGE_1'}
    # ophys_session_names = {asset_name: data_info[mouse_id]['ophys'][asset_name]['raw'] for asset_name in data_info[mouse_id]['ophys']}

    ophys_zstack_path = DATA_DIR / data_info[mouse_id]['coregistration']['ophys_zstack']
    total_match_table = pd.read_csv(list(ophys_zstack_path.glob('*corrected_fov_czstack*.csv'))[0])
    total_match_table = total_match_table.rename(columns={'resolved_cz_stack_id': 'mask_id_cz'})
    total_match_table['plane'] = total_match_table['plane_id']
    total_match_table = total_match_table[total_match_table['mask_id_cz'] >0]
    total_match_table['session'] = [s[:s.rfind('processed')-1] for s in total_match_table['session_name']]
    total_match_table['mask_id_fov'] = [int(s.split('_')[-1])+1 for s in total_match_table['unique_roi_id']]
    total_match_table['mouse_id'] = [int(s.split('_')[0]) for s in total_match_table['unique_roi_id']]
    planes = [f"VISp_{i}" for i in range(8)]
    ophys_zstack_match = []
    for plane in planes:
        for session, session_name in ophys_session_names.items():
            # cell_matching_path = ophys_zstack_path / session_name / 'cell_matching' / f"{plane}_to_zstack_match.csv"
            mask_session = total_match_table['session'] == session_name
            mask_plane = total_match_table['plane'] == plane
            df_match = total_match_table.loc[mask_plane & mask_session] 
            # df_match = pd.read_csv(cell_matching_path)[['mask_id_fov','mask_id_cz','plane','mouse_id']]
            df_match.rename(columns={'mask_id_fov': f'cell_id - {session}'}, inplace=True)
            if session == list(ophys_session_names.keys())[0]:
                df_plane = df_match.copy()
            else:
                df_plane = pd.merge(df_plane, df_match, on=['mask_id_cz','plane','mouse_id'], how='left')
        # After merging all sessions, drop rows with any NaN cell_id and cast to int
        for session in ophys_session_names:
            df_plane.dropna(subset=[f'cell_id - {session}'], inplace=True)
            df_plane[f'cell_id - {session}'] = df_plane[f'cell_id - {session}'].astype(int)
        ophys_zstack_match.append(df_plane)

    ophys_zstack_match = pd.concat(ophys_zstack_match,axis= 0)
    ophys_zstack_match.rename(columns={'mask_id_cz': 'cell_id - zstack'}, inplace=True)
    ophys_zstack_match = ophys_zstack_match[['mouse_id', 'plane', 'cell_id - zstack']+[ f'cell_id - {session}' for session in ophys_session_names]]

    hcr_zstack_match = []
    hcr_zstack_path = DATA_DIR / data_info[mouse_id]['coregistration']['hcr_zstack'] 

    cell_matching_file = list(hcr_zstack_path.glob("*_coreg_table.csv"))[-1]
    hcr_zstack_match = pd.read_csv(cell_matching_file).rename(columns={"czstack_id": "cell_id - zstack", 'hcr_id':'cell_id - hcr'},)
    hcr_zstack_match['mouse_id'] = mouse_id

    hcr_zstack_match.sort_values(['mouse_id', 'cell_id - zstack'], inplace=True)
    hcr_zstack_match = hcr_zstack_match[['mouse_id', 'cell_id - zstack', 'cell_id - hcr']]
   
    ophys_hcr_match = hcr_zstack_match.merge(ophys_zstack_match, on=['mouse_id', 'cell_id - zstack'], how='inner')
    ophys_hcr_match.insert(0, 'unique_id', ophys_hcr_match.apply(lambda row:( f"{row['mouse_id']}_{row['cell_id - zstack']}"), axis=1))
    ophys_hcr_match.to_csv(MATCH_DIR / f'{mouse_id}h_ophys_transcriptomics_match.csv', index = False)
    print(mouse_id, ophys_hcr_match.shape)


/tmp/ipykernel_2851906/2852256161.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_match.rename(columns={'mask_id_fov': f'cell_id - {session}'}, inplace=True)
/tmp/ipykernel_2851906/2852256161.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_match.rename(columns={'mask_id_fov': f'cell_id - {session}'}, inplace=True)
/tmp/ipykernel_2851906/2852256161.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_match.rename(colu

755252 (272, 8)
767018 (107, 8)


/tmp/ipykernel_2851906/2852256161.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_match.rename(columns={'mask_id_fov': f'cell_id - {session}'}, inplace=True)
/tmp/ipykernel_2851906/2852256161.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_match.rename(columns={'mask_id_fov': f'cell_id - {session}'}, inplace=True)
/tmp/ipykernel_2851906/2852256161.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_match.rename(colu

767022 (308, 8)
782149 (149, 8)


/tmp/ipykernel_2851906/2852256161.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_match.rename(columns={'mask_id_fov': f'cell_id - {session}'}, inplace=True)
/tmp/ipykernel_2851906/2852256161.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_match.rename(columns={'mask_id_fov': f'cell_id - {session}'}, inplace=True)
/tmp/ipykernel_2851906/2852256161.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_match.rename(colu

788406 (274, 8)
790322 (217, 8)


/tmp/ipykernel_2851906/2852256161.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_match.rename(columns={'mask_id_fov': f'cell_id - {session}'}, inplace=True)
/tmp/ipykernel_2851906/2852256161.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_match.rename(columns={'mask_id_fov': f'cell_id - {session}'}, inplace=True)
